# Audio Denoising using Spiking Neural Networks

Dual Degree Project implementation using an NSNet2-style architecture with spiking neural network layers.

In [ ]:
!pip install -r requirements.txt
!python setup.py install

import torch
import torch.nn as nn
from sparch.models.snns import SNN

batch_size = 4
nb_steps = 100
nb_inputs = 20
x = torch.Tensor(batch_size, nb_steps, nb_inputs)
nn.init.uniform_(x)

model = SNN(
    input_shape=(batch_size, nb_steps, nb_inputs),
    neuron_type="adLIF",
    layer_sizes=[128, 128, 10],
    normalization="batchnorm",
    dropout=0.1,
    bidirectional=False,
    use_readout_layer=False,
    )

y, firing_rates = model(x)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable parameters: {trainable_params}")

import math
import os
from pathlib import Path
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset

class WAVDataset(Dataset):
    Create a PyTorch Dataset object from a directory containing clean and noisy WAV files
    def __init__(self, dir: Path, n_fft, test=False):
        self.clean_dir = dir.joinpath('clean')
        self.noisy_dir = dir.joinpath('noisy')
        self.n_fft = n_fft
        self.test = test

        assert os.path.exists(self.clean_dir), 'No clean WAV file folder found!'
        assert os.path.exists(self.noisy_dir), 'No noisy WAV file folder found!'

        self.clean_WAVs = {}
        for i, filename in enumerate(sorted(os.listdir(self.clean_dir))):
            self.clean_WAVs[i] = self.clean_dir.joinpath(filename)

        self.noisy_WAVs = {}
        for i, filename in enumerate(sorted(os.listdir(self.noisy_dir))):
            self.noisy_WAVs[i] = self.noisy_dir.joinpath(filename)

    def __len__(self):
        return len(self.noisy_WAVs)

    def __getitem__(self, idx):
        noisy_path = self.noisy_WAVs[idx]
        clean_path = str(noisy_path).replace("noisy", "clean")
        while True:
            try:
                clean_waveform, _ = torchaudio.load(clean_path)
                noisy_waveform, _ = torchaudio.load(noisy_path)
                channels, time = noisy_waveform.shape
                clean_waveform = clean_waveform
                noisy_waveform = noisy_waveform

            except (RuntimeError, OSError):
                continue
            break

        assert clean_waveform.shape[0] == 1 and noisy_waveform.shape[0] == 1, 'WAV file is not single channel!'
        win_length = self.n_fft
        window = torch.sqrt(torch.hann_window(win_length))

        x_stft = torch.stft(noisy_waveform.view(-1), n_fft=self.n_fft, hop_length=self.n_fft // 4, win_length=self.n_fft, window=window,center = True, return_complex = True, normalized = True)
        y_stft = torch.stft(clean_waveform.view(-1), n_fft=self.n_fft, hop_length=self.n_fft // 4, win_length=self.n_fft, window=window,center = True,  return_complex = True, normalized = True)
        x_ps = x_stft.abs().pow(2)
        x_lps = LogTransform()(x_ps)

        eps = 1e-10

        if not self.test:
            return x_lps, x_stft, y_stft
        if self.test:
            return noisy_waveform, clean_waveform, x_lps, x_stft, y_stft

class LogTransform(torch.nn.Module):
    def __init__(self, floor=10**-12):
        super().__init__()
        self.floor = floor

    def forward(self, specgram):
        return torch.log(torch.clamp(specgram, min=self.floor))

import torch
import torch.nn as nn
import torch.nn.functional as F
from sparch.models.snns import SNN

batch_size = 1
nb_steps = 100
nb_inputs = 400

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class NSNet2_hybrid(nn.Module):
    def __init__(self, cfg={
            'n_fft': 508,
            'hop_len': 160,
            'win_len': 320,
        }):
        super(NSNet2_hybrid, self).__init__()
        self.n_fft = cfg['n_fft']
        self.n_freq_bins = self.n_fft // 2 + 1
        self.n_gru = 2
        self.gru_dropout = 0.2
        self.fc_input = nn.Linear(self.n_freq_bins, 400)
        self.snn1 = SNN(
                  input_shape=(batch_size, nb_steps, nb_inputs),
                  neuron_type="RadLIF",
                  layer_sizes=[128, 128, 200],
                  normalization="batchnorm",
                  dropout=0.1,
                  bidirectional=True,
                  use_readout_layer=False,
                  )
        self.snn2 = SNN(
          input_shape=(batch_size, nb_steps, nb_inputs),
          neuron_type="RadLIF",
          layer_sizes=[128, 128, 200],
          normalization="batchnorm",
          dropout=0.1,
          bidirectional=True,
          use_readout_layer=False,
          )
        self.dense0 = nn.Linear(400, 600)
        self.dense1 = nn.Linear(600, 600)
        self.dense2 = nn.Linear(600, self.n_freq_bins)

    def forward(self, x):
        x = torch.relu(self.fc_input(x))
        x, _ = self.snn1(x)
        x, _ = self.snn2(x)
        x = torch.relu(self.dense0(x))
        x = torch.relu(self.dense1(x))
        x = torch.sigmoid(self.dense2(x))
        return x

x = torch.randn(1, 100, 255)
model = NSNet2_hybrid()
y = model(x)
print('op shape',y.shape)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable parameters: {trainable_params}")

import torch
from torch import nn, optim
from tqdm import tqdm
from torch.utils.data import DataLoader
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

train_dir = Path('/content/drive/MyDrive/audio_train')
val_dir = Path('/content/drive/MyDrive/audio_val')

train_cfg = {
    'train_dir': train_dir,
    'val_dir': val_dir,
    'batch_size': 1,
    'alpha': 0.35,
}

model_cfg = {
    'n_fft': 508,
    'hop_len': 160,
    'win_len': 320,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = WAVDataset(train_cfg['train_dir'], n_fft=model_cfg['n_fft'])
val_dataset = WAVDataset(train_cfg['val_dir'], n_fft=model_cfg['n_fft'],test = True)

train_loader = DataLoader(train_dataset, batch_size=train_cfg['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=train_cfg['batch_size'], shuffle=True)


import torch
from tqdm import tqdm
import numpy as np
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

def compute_snr(clean, enhanced):
    noise = clean - enhanced
    signal_power = torch.sum(clean ** 2, dim=-1)
    noise_power = torch.sum(noise ** 2, dim=-1)
    snr = 10 * torch.log10(signal_power / (noise_power + 1e-8))
    return snr.mean().item()

def si_sdr(reference, estimation, eps=1e-8):
    reference = reference - reference.mean(dim=-1, keepdim=True)
    estimation = estimation - estimation.mean(dim=-1, keepdim=True)
    scale = torch.sum(reference * estimation, dim=-1, keepdim=True) / (torch.sum(reference ** 2, dim=-1, keepdim=True) + eps)
    projection = scale * reference
    noise = estimation - projection
    ratio = torch.sum(projection ** 2, dim=-1) / (torch.sum(noise ** 2, dim=-1) + eps)
    return 10 * torch.log10(ratio + eps)


class CompressedComplexLoss(nn.Module):
    def __init__(self, c=0.3, alpha=0.3, eps=1e-8):
        super(CompressedComplexLoss, self).__init__()
        self.c = c  # Compression exponent
        self.alpha = alpha  # Magnitude/complex mix factor
        self.eps = eps

    def forward(self, S_pred, S_true):
        S_pred: predicted complex STFT (batch, freq, time), dtype=torch.cfloat
        S_true: target complex STFT (batch, freq, time), dtype=torch.cfloat

        mag_pred = S_pred.abs()
        mag_true = S_true.abs()

        std_pred = mag_pred.std(dim=-1, keepdim=True) + self.eps
        std_true = mag_true.std(dim=-1, keepdim=True) + self.eps

        S_pred_norm = S_pred / std_pred
        S_true_norm = S_true / std_true

        compressed_S_pred = (S_pred_norm.abs().clamp(min=self.eps) ** self.c) * torch.exp(1j * S_pred_norm.angle())
        compressed_S_true = (S_true_norm.abs().clamp(min=self.eps) ** self.c) * torch.exp(1j * S_true_norm.angle())

        complex_loss = torch.abs(compressed_S_pred - compressed_S_true) ** 2

        mag_loss = F.mse_loss(mag_pred ** self.c, mag_true ** self.c)

        return self.alpha * complex_loss.sum() + (1 - self.alpha) * mag_loss

from IPython.display import Audio, display
import torchaudio

model = NSNet2_hybrid(model_cfg).to(device)
optimizer = AdamW(model.parameters(), lr=1e-4)
criterion = CompressedComplexLoss(c=0.3, alpha=0.3)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.9, patience=5, verbose=True)

epochs = 10
for epoch in range(epochs):
    model.train()

    train_loss = 0.0
    for x_lps, x_stft, y_stft in tqdm(train_loader, desc=f"[Epoch {epoch+1}] Training"):
        x_stft = x_stft.to(device)
        x_lps = x_lps.to(device)
        y_stft = y_stft.to(device)
        x_lps = x_lps.transpose(1, 2)
        x_stft = x_stft.transpose(1, 2)
        y_stft = y_stft.transpose(1, 2)

        optimizer.zero_grad()
        mask = model(x_lps)
        enhanced = x_stft * mask
        loss = criterion(enhanced, y_stft)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    scheduler.step(avg_train_loss)
    print(f"Epoch [{epoch+1}/{epochs}] - Avg Train Loss: {avg_train_loss:.6f}")

    model.eval()
    val_loss = 0.0
    snr_scores = []
    si_sdr_scores = []

    with torch.no_grad():
        for noisy_waveform, clean_waveform, x_lps, x_stft, y_stft in tqdm(val_loader, desc=f"[Epoch {epoch+1}] Validation"):
            x_lps = x_lps.to(device)
            x_stft = x_stft.to(device)
            y_stft = y_stft.to(device)
            x_lps = x_lps.transpose(1, 2)
            x_stft = x_stft.transpose(1, 2)
            y_stft = y_stft.transpose(1, 2)

            mask = model(x_lps)
            enhanced_stft = x_stft * mask

            loss = criterion(enhanced_stft, y_stft)
            val_loss += loss.item()

            window = torch.sqrt(torch.hann_window(model_cfg['n_fft'])).to(device)
            enhanced_waveform = torch.istft(enhanced_stft.squeeze(0).transpose(0, 1), n_fft=model_cfg['n_fft'], hop_length=model_cfg['n_fft'] // 4, win_length=model_cfg['n_fft'], window=window, center = True, normalized = True, return_complex=False, length = clean_waveform.shape[-1])
            clean_waveform = clean_waveform.squeeze(0).squeeze(0).to(device)

            enhanced_np = enhanced_waveform.cpu().numpy()
            clean_np = clean_waveform.cpu().numpy()


            snr_score = compute_snr(clean_waveform, enhanced_waveform)
            snr_scores.append(snr_score)

            si_sdr_score = si_sdr(clean_waveform.unsqueeze(0), enhanced_waveform.unsqueeze(0))
            si_sdr_scores.append(si_sdr_score.item())

    avg_val_loss = val_loss / len(val_loader)
    avg_snr = np.mean(snr_scores)
    avg_si_sdr = np.mean(si_sdr_scores)

        print(f"Validation Loss: {avg_val_loss:.6f} | SNR: {avg_snr:.2f} dB | SI-SDR: {avg_si_sdr:.2f} dB")
